# **Stage 1: Comprehensive geospatial and operational inventory of warehouses**


RQ1. What are the warehouses and distribution center’s location patterns over time and space?

The first phase involves constructing a comprehensive geospatial and operational inventory of warehouses. The inventory will include key variables such as warehouse location, building footprint (in square footage), land use type, and operational intensity (measured by truck trip counts).

To enhance analytical precision, the team will also develop a typology of warehouse facilities based on building size, proximity to major freight corridors, and distance to various communities, as defined by CalEnviroScreen or equivalent indicators. 

Finally, the inventory will be geocoded and integrated into a GIS platform, enabling spatial analysis, overlay with environmental justice indicators, and linkage to emissions modeling in subsequent stages.



## Libraries

In [ ]:
# Importing all the required packages
# Python 3.11.9

# Basic
import pandas as pd # pandas 2.3.2
import geopandas as gdp # geopandas 0.14.3
import numpy as np # nympy 1.26.4
import matplotlib.pyplot as plt # matplolib 3.10.6
import statsmodels.api as sm
from scipy import stats
from shapely.ops import unary_union
import folium
import fiona, pyproj  # fiona 1.9.6. pyproj 3.7.2
import requests
from shapely.geometry import box # shapely 2.1.2
from io import BytesIO #In-Memory Binary Stream
import tempfile
import zipfile
import subprocess
import gzip # request from GitHub
import shutil
import time
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
import json
import re
from pathlib import Path

# OpenStreetMaps
import osmnx as ox
import os

# Overturemaps
import overturemaps as om

# Microsoft Building
from pystac_client import Client
import planetary_computer




path = 'YOUR_PATH_HERE' # Set this to your local path for data storage

In [2]:
# Earth Engine
import ee
import geemap

# Authenticate if first time - GEE
ee.Authenticate(auth_mode='notebook')
# Initialize
ee.Initialize()

## **Process to find warehouses ≥100,000 ft²**

Step 1. Define the Temporal Resolution and Study Area
- Timeframe: 2021-2023
- Frequency: Annual 
- Area: LA, Orange, Riverside, San Bernardino boundary 
- CRS: EPSG:2229 (CA State Plane CA-V, feet) or EPSG:3310 (meters; convert to ft²).

Step 2. Collect primary footprint sources
- USDA NAIP aerial (~0.6 m GSD) 2021 - 2023 (This can help to complete buildings inventory)
- Overture Maps – Buildings: consolidated OSM/Microsoft/Google/Esri sources in one schema; polygons are footprints/roofprints. Great coverage & deduplication. 
- Microsoft US/Global Building Footprints (GeoJSON/Parquet via Microsoft Planetary Computer). US set derived from high-res imagery; huge coverage. 
- OpenStreetMaps buildings for building information
Note: Overture/Microsoft polygons sometimes lack per-feature acquisition dates

Step 3. Spatial join and building classification
- Combine Overture + Microsoft + OSM + NAIP and check polygon areas

Step 4. Land use / zoning tag (for each warehouse site)
- Source: OSM and SCAG’s parcel-level land-use layers (ALU 2019/2016) and/or local general plan/zoning layers

Step 5. Truck trip data (operational intensity)
- South Coast AQMD’s WAIRE
- PeMS classification + WIM stations: vehicle class shares/weights where available (historical archive).
- SCAG High-Cube Warehouse trip-rate study (CEQA): published trip rates per 1,000 ft² for high-cube facilities; use with caution and cite assumptions until AWR comes.

Step 6. Validation and integration 
- Join source polygon (Overture/Microsoft/OSM), imagery years checked (2021/2022/2023), parcel cross-ref, land-use tag, truck trip sources (AWR/Caltrans/PeMS/CEQA).
- For geometry QA, flag odd shapes (L-shaped, fragmented roofs) and campuses where multiple roofs form a single warehouse operation (DBSCAN to dissolve)

Step 7: Warehouse typology

Deliverables:
- waire_inventory_sites.gpkg (sites, areas, temporal flags, land use, truck metrics, typology).
- methods.md detailing sources/assumptions (especially for trip estimates and the single-building vs. campus aggregation rule)

### Step 1: Define the Study Area

In [3]:
# Define a function to get ROI by county and state FIPS
def get_roi(county_names,
                         state_fips='06',
                         county_dataset='TIGER/2018/Counties'):
    """
    Returns a single ee.Geometry covering multiple counties in a state.

    Parameters
    ----------
    county_names : list of str
        List of county names (e.g., ['Los Angeles','Orange','Riverside','San Bernardino'])
    state_fips : str
        State FIPS code (default '06' for California)
    county_dataset : str
        TIGER/Counties FeatureCollection path

    Returns
    -------
    ee.Geometry
        Merged geometry (union) of all specified counties
    """
    counties = ee.FeatureCollection(county_dataset)

    # filter by state first
    state_counties = counties.filter(ee.Filter.eq('STATEFP', state_fips))

    # merge geometries for all selected counties
    roi = None
    for c in county_names:
        geom = state_counties.filter(ee.Filter.eq('NAME', c)).geometry()
        roi = geom if roi is None else roi.union(geom)

    return roi



### Step 2: Collect primary footprint sources

| Dataset                 | Strength                                                       | Limitation                                       |
| ----------------------- | -------------------------------------------------------------- | ------------------------------------------------ |
| **NAIP**                | Includes all the buildings (very complete source)              | No building information, requires post processing|
| **Microsoft Buildings** | Very accurate roof geometry, good for size estimation          | No semantics or “type” info                      |
| **Overture Buildings**  | Includes some `class` or `subclass` + OSM attributes (`name`)  | Still misses industrial parks or complex parcels |
| **OSM Buildings**       | Has operator names, labels (e.g., “Amazon Fulfillment Center”) | Sparse in many areas; incomplete coverage        |


These sources are:

* legally allowed
* reproducible
* transparent
* scientifically defensible
* suitable for statewide scaling

#### 2.1. USDA NAIP

In [4]:
# Get the NAIP data
def get_naip_image(roi, ini_date='2022-01-01', end_date='2022-12-31'):
    """
    Retrieve a NAIP mosaic clipped to the ROI between given dates.
    """
    naip = (ee.ImageCollection('USDA/NAIP/DOQQ')
              .filterBounds(roi)
              .filterDate(ini_date, end_date)
              .mosaic()
              .clip(roi))
    return naip


def detect_large_roofs(roi, start_date='2022-01-01', end_date='2022-12-31',
                       area_threshold_m2=9290, scale=3):
    """
    Detect large built-up roof surfaces (>~100k ft²) from NAIP imagery.

    Parameters
    ----------
    roi : ee.Geometry or ee.FeatureCollection
        Region of interest.
    start_date, end_date : str
        NAIP date range (e.g., '2022-01-01', '2022-12-31').
    area_threshold_m2 : float
        Minimum area threshold in m² (default 9290 ≈ 100,000 ft²).
    scale : int
        Pixel scale for reduceToVectors (default 1 m).

    Returns
    -------
    dict
        {
          'naip': ee.Image,
          'roof_mask': ee.Image,
          'roof_polygons': ee.FeatureCollection
        }
    """

    # 1️⃣ Mosaic NAIP imagery
    naip = get_naip_image(roi, start_date, end_date)

    # 2️⃣ Compute indices (NAIP: R,G,B,N)
    ndvi = naip.normalizedDifference(['N', 'R']).rename('NDVI')
    ndbi = naip.normalizedDifference(['G', 'N']).rename('NDBI')  # built-up proxy

    # 3️⃣ Roof-like mask: bright, non-vegetated, impervious
    roof_mask = ndbi.gt(0.0).And(ndvi.lt(0.4)).selfMask()

    # 4️⃣ Keep large connected regions
    roof_connected = roof_mask.connectedPixelCount(50, True).gte(20).selfMask()

    # 5️⃣ Vectorize polygons
    vectors = roof_connected.reduceToVectors(
        geometry=roi,
        scale=scale,
        geometryType='polygon',
        eightConnected=True,
        labelProperty='roof',
        bestEffort=True,
        maxPixels=1e9
    )

    # 6️⃣ Add area property and filter ≥ threshold
    vectors = vectors.map(
    lambda f: f.set('area_m2', ee.Geometry(f.geometry()).area())
    )
    roof_polygons = vectors.filter(ee.Filter.gt('area_m2', area_threshold_m2))

    # return dictionary of results
    print("Vector count: ", roof_polygons.size().getInfo())
    #print(f"Returned {len(large_buildings):,} buildings ≥ {area_threshold_ft2:,} ft² from Overture")
    return {'naip': naip, 'roof_mask': roof_connected, 'roof_polygons': roof_polygons}


#### 2.2. Overture Maps

In [5]:
# Download overture data 
#https://docs.overturemaps.org/guides/buildings/#14/32.58453/-117.05154/0/60

# Helpers
def tile_bbox(bbox, nx=4, ny=4):
    minx, miny, maxx, maxy = bbox
    xs = np.linspace(minx, maxx, nx+1)
    ys = np.linspace(miny, maxy, ny+1)
    tiles = []
    for i in range(nx):
        for j in range(ny):
            tiles.append([
                xs[i], ys[j], xs[i+1], ys[j+1]
            ])
    return tiles

def download_overture_tiled(county_prefix, bbox, nx=4, ny=4):
    tiles = tile_bbox(bbox, nx, ny)
    out_files = []
    
    for k, tile in enumerate(tiles):
        bbox_str = ",".join(map(str, tile))
        out_file = f"{county_prefix}_tile{k}.geojson"
        cmd = [
            "overturemaps", "download",
            "--type=building",
            "--bbox", bbox_str,
            "-f", "geojson",
            "-o", out_file
        ]

        print(f"⬇️ Downloading tile {k} for {county_prefix} → {out_file}")
        try:
            subprocess.run(cmd, check=True)
            out_files.append(out_file)
        except Exception as e:
            print(f"❌ Tile {k} failed: {e}")
        
        time.sleep(1)  # avoid rate limit
    
    return out_files

def merge_overture_tiles(tile_files):
    gdfs = []
    for file in tile_files:
        try:
            gdf = gdp.read_file(file)
            gdfs.append(gdf)
        except:
            pass
    if not gdfs:
        print("❌ No tiles loaded.")
        return None
    merged = gdp.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=4326)
    print(f"✅ Merged {len(tile_files)} tiles → {len(merged)} features")
    return merged

# Alternative but sometimes with errors
def run_overture_download(county_name, bbox):
    bbox_str = ",".join(map(str, bbox))
    out_file = f"{county_name.lower()}_buildings.geojson"
    cmd = [
        "overturemaps", "download",
        "--type=building",
        "--bbox", bbox_str,
        "-f", "geojson",
        "-o", out_file
    ]
    print(f"Downloading {county_name} buildings...")
    subprocess.run(cmd, check=True)
    print(f"Saved to {out_file}\n")

def get_overture_building(buildings_path, area_threshold_ft2=100_000, county="None"):
    """
    Loads Overture Buildings/Places and filters industrial or large buildings.

    Parameters
    ----------
    buildings_path : str
        Path to local Overture GeoParquet or GeoJSON file.
    area_threshold_ft2 : float, optional
        Minimum building footprint area (in square feet). Default = 100,000.
    county : str
        County name

    Returns
    -------
    gpd.GeoDataFrame
        Filtered GeoDataFrame of buildings ≥ threshold area.
    """
    try:
        with fiona.Env():
            gdf = gdp.read_file(buildings_path)
    except Exception as e:
        raise RuntimeError(f"Failed to read {buildings_path}: {e}")

    # Ensure we have geometries
    gdf = gdf[gdf.geometry.notnull()].copy()

    # Reproject to a projected CRS suitable for area (California Albers)
    gdf = gdf.to_crs(epsg=3310)

    # Compute footprint area
    gdf["area_m2"] = gdf.geometry.area
    gdf["area_ft2"] = gdf["area_m2"] * 10.7639

    # Filter by area threshold
    large_buildings = gdf.loc[gdf["area_ft2"] >= area_threshold_ft2].copy()

    # Back to WGS84 (export or visualize in GEE/Leaflet)
    large_buildings = large_buildings.to_crs(epsg=4326)
    
    print(county)
    print(f"Returned {len(large_buildings):,} buildings ≥ {area_threshold_ft2:,} ft² from Overture")
    
    return large_buildings


#### 2.3. Microsoft US/Global Building Footprints

In [6]:
def get_ms_buildings(bbox, state_name="California", area_threshold_ft2=100_000,
                     cache_dir="data"):
    """
    Downloads Microsoft US Building Footprints within a given bounding box.

    Parameters
    ----------
    bbox : list [min_lon, min_lat, max_lon, max_lat]
        Geographic bounding box (EPSG:4326)
    state_abbr : str, optional
        State abbreviation (default 'CA')
    area_threshold_ft2 : float, optional
        Minimum building footprint area (in square feet). Default = 100,000.
        
    Returns
    -------
    gdp.GeoDataFrame
        GeoDataFrame with building footprints inside bbox
    """
    os.makedirs(cache_dir, exist_ok=True)
    zip_path = os.path.join(cache_dir, f"{state_name}.geojson.zip")
    
    # Microsoft building footprints (US) are on GitHub / Azure Blob:
    base_url = f"https://minedbuildings.z5.web.core.windows.net/legacy/usbuildings-v2/{state_name}.geojson.zip"
    
    
    # -------- 1. Download if not cached  --------
    if not os.path.exists(zip_path):
        print(f"⬇️  Downloading {state_name} footprints (~4–5 GB)...")
        session = requests.Session()
        retry = Retry(total=5, backoff_factor=5,
                      status_forcelist=[500, 502, 503, 504],
                      allowed_methods=["GET"])
        session.mount("https://", HTTPAdapter(max_retries=retry))
        with session.get(base_url, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(zip_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=10_485_760):  # 10 MB
                    if chunk:
                        f.write(chunk)
        print(f"✅  Saved to {zip_path}")
    else:
        print(f"✅  Using cached file {zip_path}")
    

    # -------- 2. Extract & read only bbox --------
    with tempfile.TemporaryDirectory() as tmpdir:
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(tmpdir)
            geojson_files = [f for f in zip_ref.namelist() if f.endswith(".geojson")]
            if not geojson_files:
                raise FileNotFoundError("No GeoJSON inside ZIP")
            geojson_path = os.path.join(tmpdir, geojson_files[0])

        print(f"📂 Reading {geojson_files[0]} subset for bbox {bbox} ...")
        gdf = gdp.read_file(geojson_path, bbox=bbox)

    # -------- 3. Compute area & filter --------
    gdf = gdf.to_crs(3310)
    gdf["area_ft2"] = gdf.area * 10.7639
    gdf = gdf[gdf["area_ft2"] >= area_threshold_ft2].to_crs(4326)
    print(f"✅  Returned {len(gdf):,} buildings ≥ {area_threshold_ft2:,} ft² from Microsoft")
    return gdf

#### 2.4. OSM (buildings tags)

In [7]:
ox.settings.timeout = 600

def get_osm_buildings(place_name, download_year=None, area_threshold_ft2=100_000):
    """
    Downloads OSM buildings polygons for a given place.

    Parameters
    ----------
    place_name : str
        Place name (e.g., 'Sacramento, California, USA')
    download_year : int, optional
        Year of download (added as 'source_year' column)

    Returns
    -------
    gpd.GeoDataFrame
        GeoDataFrame of OSM buildings polygons
    """
    # Default parameters https://wiki.openstreetmap.org/wiki/Key:building
    tags = {
            'building':   ['industrial', 'warehouse', 'commercial', 
                           'supermarket', 'retail', 'office','museum', 
                           'school', 'university', 'sports_centre', 
                           'stadium', 'military', 'hospital', 'church',
                           'cathedral', 'hotel', 'transportation', 
                           'parking', 'apartments', 'residential'],
            'amenity':    ['cinema', 'casino', 'theatre', 'prison',
                           'police', 'ranger_station', 'fire_station',
                           'clinic', 'doctors', 'nursing_home',
                           'library', 'planetarium', 'arts_centre',
                           'community_centre', 'conference_centre',
                           'kindergarten', 'school', 'university',
                           'college', 'pharmacy', 'veterinary',
                           'social_facility', 'social_centre',
                           'shelter', 'marketplace'],
            'tourism':    ['zoo', 'aquarium', 'theme_park', 'hotel',
                           'motel', 'guest_house', 'chalet'],
            'leisure':    ['water_park', 'resort'],
            'shop':       ['supermarket', 'convenience', 'general',
                           'clothes', 'boutiqueboutique', 
                           'electronics', 'wholesale'],
            'healthcare': ['hospital', 'clinic', 'doctor', 
                           'dentist', 'physiotherapist']
        }
    if download_year is None:
        from datetime import datetime
        download_year = datetime.now().year

    # -------- 1. OSMnx unified function for geometries (2.x) or fallback for 1.x
    if hasattr(ox, "features_from_place"):
        gdf = ox.features_from_place(place_name, tags=tags)
    else:
        gdf = ox.geometries_from_place(place_name, tags=tags)

    # --------- 2. Adjust properties ----------
    # Add source year
    gdf["source_year"] = download_year

    # Keep only polygons
    gdf = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon", "Point"])]


    # -------- 3. Compute area & filter --------
    #gdf = gdf.to_crs(3310)
    #gdf["area_ft2"] = gdf.geometry.area * 10.7639

    # Filter
    #gdf = gdf[gdf["area_ft2"] >= area_threshold_ft2].copy()
    #print(f"✅  Returned {len(gdf):,} buildings ≥ {area_threshold_ft2:,} ft² from OSM")
    #print("Unique buildings types:", gdf["building"].value_counts().to_dict())
    
    columns_to_keep = ['geometry', 'name', 'building', 'tourism', 'amenity', 
                       'shop', 'healthcare', 'source_year']
    gdf_reduced = gdf[columns_to_keep].copy()
    
    # --------- 4. Reset CRS to WGS84 for consistency
    gdf_reduced = gdf_reduced.to_crs(epsg=4326)
    return gdf_reduced


### Step 3: Spatial join and classification

#### 3.1. Overlap ratio

In [8]:
def overlap_ratio(row, ms_df):
    # No matching MS building
    idx = row.index_right
    if pd.isna(idx):
        return 0
    
    # Ensure index exists
    if idx not in ms_df.index:
        return 0
    
    geom1 = row.geometry
    geom2 = ms_df.loc[idx].geometry
    
    # Validate geometries
    if geom1 is None or geom1.is_empty:
        return 0
    if geom2 is None or geom2.is_empty:
        return 0
    
    # Check intersection safely
    if not geom1.intersects(geom2):
        return 0
    
    inter = geom1.intersection(geom2)
    if inter.is_empty:
        return 0
    
    union = geom1.union(geom2)
    if union.is_empty:
        return 0
    
    # Compute ratio
    return inter.area / union.area if union.area > 0 else 0


#### 3.2. Building selection based on keywords and shape metrics




In [ ]:

# ==============================================================================
# LOOKUP TABLES
# ==============================================================================

# --- Overture subtype values → EXCLUDE ---
EXCLUDE_SUBTYPES = {
    'education', 'residential', 'accommodation',
    'entertainment', 'civic', 'medical', 'religious'
}

# --- Overture class values → EXCLUDE ---
EXCLUDE_CLASSES = {
    'hotel', 'school', 'hospital', 'apartments', 'parking',
    'retail', 'supermarket', 'office', 'church', 'government',
    'fire_station', 'police', 'stadium', 'museum', 'prison',
    'courthouse', 'library', 'kindergarten', 'college',
    'university', 'nursing_home', 'clinic', 'pharmacy',
    'veterinary', 'motel', 'guest_house', 'hostel',
    'sports_centre', 'water_park', 'theme_park', 'zoo',
    'aquarium', 'cinema', 'theatre', 'casino', 'shelter',
    'community_centre', 'arts_centre', 'marketplace'
}

# --- Overture class values → CONFIRM industrial/warehouse ---
CONFIRM_CLASSES = {
    'warehouse', 'industrial', 'logistics', 'storage',
    'manufacturing', 'data_center', 'utility', 'factory',
    'distribution', 'cold_storage', 'fulfillment'
}

# --- OSM building tags → EXCLUDE ---
EXCLUDE_OSM_TAGS = set([
    'commercial', 'supermarket', 'retail', 'office', 'museum',
    'school', 'university', 'sports_centre', 'stadium', 'military',
    'hospital', 'church', 'cathedral', 'hotel', 'transportation',
    'parking', 'apartments', 'residential', 'cinema', 'casino',
    'theatre', 'prison', 'police', 'ranger_station', 'fire_station',
    'clinic', 'doctors', 'nursing_home', 'zoo', 'aquarium',
    'theme_park', 'library', 'planetarium', 'arts_centre',
    'community_centre', 'conference_centre', 'kindergarten', 'college',
    'pharmacy', 'veterinary', 'social_facility', 'social_centre',
    'shelter', 'marketplace', 'motel', 'guest_house', 'chalet',
    'water_park', 'resort', 'convenience', 'general', 'clothes',
    'electronics', 'wholesale', 'dentist', 'physiotherapist',
    'highschool', 'place_of_worship', 'car_wash', 'townhall',
    'courthouse', 'embassy', 'government', 'public_building', 'car_repair'
])

# ------------------------------------------------------------------------------
# NAME-BASED EXCLUSION KEYWORDS
# These are non-warehouse land uses that may lack subtype/class tags in Overture.
# Using partial lowercase match — order matters for specificity.
# ------------------------------------------------------------------------------
EXCLUDE_NAME_KEYWORDS = [
    # --- Healthcare ---
    'hospital', 'medical center', 'medical centre', 'health system',
    'kaiser permanente', 'dignity health', 'torrance memorial',
    'cedars-sinai', 'providence', 'urgent care', 'surgery center',
    'dialysis', 'rehabilitation center', 'convalescent',

    # --- Retail storefronts (large footprint but NOT WAIRE DCs) ---
    # NOTE: "walmart dc", "target dc" etc. are intentionally NOT excluded
    # because DCs ≥100k ft² ARE subject to Rule 2305.
    # We only exclude storefront identifiers here.
    'home depot', "lowe's", 'lowes', 'ikea', 'best buy',
    "sam's club", 'big lots', 'smart & final', 'grocery outlet',
    'costco wholesale',          # storefront; costco DCs handled separately
    'walmart supercenter',       # storefront only
    'walmart neighborhood',      # storefront only
    'target store',              # storefront only

    # --- Civic / Government ---
    'city hall', 'county courthouse', 'fire station', 'fire house',
    'police department', 'police station', 'sheriff', 'post office',
    'dmv ', 'department of motor', 'united states postal',
    'federal building', 'city of ', 'county of ',

    # --- Transportation terminals (not freight warehouses) ---
    'passenger terminal', 'airport terminal', 'cruise terminal',
    'amtrak station', 'metro station', 'transit center',
    'bus depot',                 # passenger, not freight

    # --- Education ---
    'high school', 'middle school', 'elementary school',
    'unified school', 'school district', 'university of',
    'college of', 'community college', 'trade school',

    # --- Hotels / Hospitality ---
    'marriott', 'hilton', 'hyatt', 'sheraton', 'holiday inn',
    'hampton inn', 'courtyard by', 'residence inn', 'extended stay',
    'best western', 'la quinta',

    # --- Worship ---
    'church', 'temple', 'mosque', 'cathedral', 'faith community',
    'ministries', 'parish', 'synagogue', 'kingdom hall',

    # --- Entertainment / Sport ---
    'stadium', 'arena', 'convention center', 'convention centre',
    'event center', 'amphitheater', 'movie theater', 'cinemark',
    'amc theater', 'regal cinema',

    # --- Residential ---
    'apartment', 'senior living', 'assisted living',
    'condominiums', 'townhomes',
]

# ------------------------------------------------------------------------------
# NAME-BASED INCLUSION KEYWORDS
# Positive signals that confirm a warehouse / distribution / industrial use.
# These are checked AFTER exclusions, so a DC named "Target Distribution"
# will not be excluded by the storefront rules above.
# ------------------------------------------------------------------------------
INCLUDE_NAME_KEYWORDS = [
    # --- Explicit function labels ---
    'fulfillment center', 'fulfillment centre',
    'distribution center', 'distribution centre',
    'distribution hub', 'sortation center', 'sort center',
    'returns center', 'delivery station',
    'cross-dock', 'crossdock', 'transload',
    'cold storage', 'refrigerated warehouse', 'frozen storage',

    # --- Major e-commerce / retail DCs (Rule 2305 targets) ---
    'amazon',                    # covers Amazon FC, Amazon DSP, Amazon Fresh DC
    'walmart dc', 'walmart distribution', 'walmart fulfillment',
    'target dc', 'target distribution', 'target fulfillment',
    'home depot distribution', 'home depot dc',
    'costco distribution', 'costco depot',
    'ikea distribution',
    "lowe's distribution", 'lowes distribution',

    # --- Parcel / freight carriers ---
    'fedex', 'ups ', 'united parcel service',
    'dhl', 'ontrac', 'lasership', 'usps distribution',
    'usps processing',

    # --- 3PL / trucking operators ---
    'xpo logistics', 'ryder ', 'penske logistics',
    'schneider national', 'estes express', 'old dominion',
    'saia ', 'averitt', 'dayton freight', 'pilot freight',
    'kuehne', 'geodis', 'ceva logistics', 'db schenker',
    'nippon express', 'kintetsu', 'yusen logistics',

    # --- Industrial real estate / park names ---
    'prologis', 'industrial park', 'business park',
    'commerce park', 'trade center', 'industrial center',
    'logistics park', 'logistics center', 'intermodal',
    'corporate park', 'commerce center',

    # --- SoCal port / intermodal operators ---
    'maersk', 'trapac', 'everport', 'apm terminal',
    'yang ming', 'evergreen logistics',

    # --- Food & beverage distribution (common in IE) ---
    'niagara bottling', 'stater bros', 'sysco', 'us foods',
    'performance food', 'gordon food', 'shamrock foods',
    'reyes holdings', 'core-mark', 'mclane',

    # --- Manufacturing signals ---
    'manufacturing', 'fabrication', 'assembly plant',
    'bottling plant', 'packaging facility',
]


# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================

def parse_primary_name(name_field):
    """
    Safely extract the 'primary' name from Overture's names field.
    Handles: dict, JSON string, plain string, or None/NaN.
    """
    if name_field is None:
        return None
    if isinstance(name_field, float) and np.isnan(name_field):
        return None
    if isinstance(name_field, dict):
        return name_field.get('primary', None)
    if isinstance(name_field, str):
        try:
            # Overture sometimes uses single quotes — normalize to double
            cleaned = name_field.replace("'", '"').replace('None', 'null')
            parsed = json.loads(cleaned)
            return parsed.get('primary', None)
        except Exception:
            # Fallback: return raw string if not valid JSON
            return name_field.strip()
    return None


def name_matches(name_str, keyword_list):
    """
    Case-insensitive partial match of name_str against any keyword in list.
    Returns the matching keyword (truthy) or None (falsy).
    """
    if not name_str:
        return None
    name_lower = name_str.lower()
    for kw in keyword_list:
        if kw in name_lower:
            return kw
    return None


def compute_shape_metrics(gdf):
    """
    Adds geometric shape metrics used to classify candidate buildings.

    Metrics added:
    - compactness     : 4π·area / perimeter² (1.0 = perfect circle)
    - rectangularity  : area / MBR area (1.0 = perfect rectangle)
    - elongation      : MBR long side / short side (higher = more elongated)

    Warehouse heuristics (approximate thresholds):
    - rectangularity > 0.80  (warehouses are boxy)
    - elongation     > 2.0   (cross-docks tend to be elongated)
    - compactness    < 0.60  (large irregular shapes → unlikely warehouse)
    """
    gdf = gdf.copy().to_crs(3310)
    gdf['area_m2']    = gdf.geometry.area
    gdf['perimeter_m'] = gdf.geometry.length

    gdf['compactness'] = (4 * np.pi * gdf['area_m2']) / (gdf['perimeter_m'] ** 2)

    def _mbr_metrics(geom):
        try:
            mbr = geom.minimum_rotated_rectangle
            mbr_area = mbr.area
            coords = list(mbr.exterior.coords)
            sides = sorted([
                ((coords[i][0] - coords[i+1][0])**2 +
                 (coords[i][1] - coords[i+1][1])**2) ** 0.5
                for i in range(4)
            ])
            rect = geom.area / mbr_area if mbr_area > 0 else np.nan
            elong = sides[-1] / sides[0] if sides[0] > 0 else np.nan
            return rect, elong
        except Exception:
            return np.nan, np.nan

    metrics = gdf.geometry.apply(_mbr_metrics)
    gdf['rectangularity'] = metrics.apply(lambda x: x[0])
    gdf['elongation']     = metrics.apply(lambda x: x[1])

    return gdf.to_crs(4326)


def _is_excluded_by_osm_tags(row, tag_columns):
    """Check if any OSM tag column contains an exclusion value."""
    for col in tag_columns:
        val = row.get(col)
        if isinstance(val, str) and val in EXCLUDE_OSM_TAGS:
            return True
    return False


# ==============================================================================
# MAIN CLASSIFIER
# ==============================================================================

def industrial_buildings(osm, ov, ms, buffer_m=20):
    """
    Classify industrial/warehouse buildings using OSM, Overture, and Microsoft
    footprint sources, enriched with semantic tags and name/brand signals.

    Classification labels
    ---------------------
    'confirmed'   : Strong positive signal (OSM tag, Overture class, or known
                    operator name). High confidence warehouse/industrial.
    'candidate'   : Passes all exclusion filters but lacks positive signals.
                    Geometric shape metrics (rectangularity, elongation) should
                    be reviewed to accept or reject these manually.
    'excluded'    : Matched an exclusion rule (tag, subtype/class, or name).
                    Retained in output with label for audit purposes.

    Parameters
    ----------
    osm      : GeoDataFrame  OSM buildings (polygons + points, EPSG:4326)
    ov       : GeoDataFrame  Overture buildings (EPSG:4326)
    ms       : GeoDataFrame  Microsoft footprints (EPSG:4326)
    buffer_m : int           Buffer radius (metres) for OSM point tag carriers

    Returns
    -------
    GeoDataFrame with columns:
        source, label, confidence_note, primary_name,
        compactness, rectangularity, elongation
    """

    # -----------------------------------------------------------------------
    # 1. SPLIT OSM INTO POLYGONS AND POINTS
    # -----------------------------------------------------------------------
    osm_poly = osm[osm.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    osm_pts  = osm[osm.geom_type == "Point"].copy()

    # -----------------------------------------------------------------------
    # 2. CONFIRMED OSM INDUSTRIAL POLYGONS
    # -----------------------------------------------------------------------
    industrial_tags = ['industrial', 'warehouse']
    osm_confirmed = osm_poly[osm_poly['building'].isin(industrial_tags)].copy()
    osm_confirmed['label']           = 'confirmed'
    osm_confirmed['confidence_note'] = 'osm_building_tag'
    osm_confirmed['source']          = 'osm'
    osm_confirmed['primary_name']    = osm_confirmed.get('name', None)

    # -----------------------------------------------------------------------
    # 3. BUFFER OSM POINTS → TAG CARRIERS FOR MS / OVERTURE ENRICHMENT
    # -----------------------------------------------------------------------
    if len(osm_pts) > 0:
        osm_pts = osm_pts.to_crs(3310)
        osm_pts["geometry"] = osm_pts.geometry.buffer(buffer_m)
        osm_pts = osm_pts.to_crs(4326)

    tag_columns = ['building', 'amenity', 'tourism', 'leisure', 'shop', 'healthcare']
    existing_cols = [c for c in tag_columns if c in osm_pts.columns] if len(osm_pts) > 0 else []

    # -----------------------------------------------------------------------
    # 4. ENRICH OVERTURE WITH OSM POINT TAGS (if available)
    # -----------------------------------------------------------------------
    ov_work = ov.copy()

    if len(osm_pts) > 0 and len(existing_cols) > 0:
        ov_proj      = ov_work.to_crs(3310)
        osm_pts_proj = osm_pts.to_crs(3310)
        ov_join = gdp.sjoin(
            ov_proj,
            osm_pts_proj[existing_cols + ["geometry"]],
            how="left", predicate="intersects"
        ).drop(columns=["geometry_right"], errors="ignore")
        ov_work = ov_join.to_crs(4326)

    # -----------------------------------------------------------------------
    # 5. PARSE OVERTURE NAME FIELD
    # -----------------------------------------------------------------------
    ov_work['primary_name'] = ov_work['names'].apply(parse_primary_name)

    # -----------------------------------------------------------------------
    # 6. CLASSIFY EACH OVERTURE BUILDING
    # -----------------------------------------------------------------------
    def _classify_overture(row):
        """
        Returns (label, confidence_note) for a single Overture row.

        Order of precedence:
        1. Positive name keyword  → confirmed  (before subtype/class exclusion
                                                so DCs are never excluded)
        2. Subtype / class exclusion → excluded
        3. OSM tag exclusion (from point join) → excluded
        4. Name exclusion keyword  → excluded
        5. Confirmed class         → confirmed
        6. Fallthrough             → candidate
        """
        subtype = row.get('subtype') or ''
        cls     = row.get('class')   or ''
        name    = row.get('primary_name') or ''

        # --- Step 1: Positive name match wins over everything ---
        pos_kw = name_matches(name, INCLUDE_NAME_KEYWORDS)
        if pos_kw:
            return 'confirmed', f'name_keyword:{pos_kw}'

        # --- Step 2: Overture subtype / class exclusion ---
        if subtype in EXCLUDE_SUBTYPES:
            return 'excluded', f'overture_subtype:{subtype}'
        if cls in EXCLUDE_CLASSES:
            return 'excluded', f'overture_class:{cls}'

        # --- Step 3: OSM tag exclusion (from buffered point join) ---
        if _is_excluded_by_osm_tags(row, existing_cols):
            hit = next(
                (row.get(c) for c in existing_cols
                 if isinstance(row.get(c), str) and row.get(c) in EXCLUDE_OSM_TAGS),
                'unknown'
            )
            return 'excluded', f'osm_tag:{hit}'

        # --- Step 4: Name exclusion keyword ---
        neg_kw = name_matches(name, EXCLUDE_NAME_KEYWORDS)
        if neg_kw:
            return 'excluded', f'name_keyword_exclude:{neg_kw}'

        # --- Step 5: Confirmed Overture class ---
        if cls in CONFIRM_CLASSES:
            return 'confirmed', f'overture_class:{cls}'

        # --- Step 6: No signal either way ---
        return 'candidate', 'no_signal'

    results = ov_work.apply(_classify_overture, axis=1, result_type='expand')
    results.columns = ['label', 'confidence_note']
    ov_work = pd.concat([ov_work, results], axis=1)
    ov_work['source'] = 'overture'

    # -----------------------------------------------------------------------
    # 7. ENRICH MICROSOFT FOOTPRINTS WITH OSM POINT TAGS + CLASSIFY
    # -----------------------------------------------------------------------
    ms_work = ms.copy()

    if len(osm_pts) > 0 and len(existing_cols) > 0:
        ms_proj = ms_work.to_crs(3310)
        ms_join = gdp.sjoin(
            ms_proj,
            osm_pts_proj[existing_cols + ["geometry"]],
            how="left", predicate="intersects"
        ).drop(columns=["geometry_right"], errors="ignore")
        ms_work = ms_join.to_crs(4326)

    # MS has no name or subtype — only OSM tag exclusion applies
    ms_work['primary_name'] = None

    def _classify_ms(row):
        if _is_excluded_by_osm_tags(row, existing_cols):
            hit = next(
                (row.get(c) for c in existing_cols
                 if isinstance(row.get(c), str) and row.get(c) in EXCLUDE_OSM_TAGS),
                'unknown'
            )
            return 'excluded', f'osm_tag:{hit}'
        return 'candidate', 'ms_no_semantic_tag'

    ms_results = ms_work.apply(_classify_ms, axis=1, result_type='expand')
    ms_results.columns = ['label', 'confidence_note']
    ms_work = pd.concat([ms_work, ms_results], axis=1)
    ms_work['source'] = 'ms'

    # -----------------------------------------------------------------------
    # 8. COMBINE ALL SOURCES
    #    Keep ALL labels in output — exclusions retained for audit trail.
    #    Drop exact geometry duplicates.
    # -----------------------------------------------------------------------
    final = pd.concat(
        [osm_confirmed, ov_work, ms_work],
        ignore_index=True
    ).drop_duplicates(subset='geometry')

    # -----------------------------------------------------------------------
    # 9. SHAPE METRICS (applied to confirmed + candidate only for speed,
    #    but computed on all so excluded buildings can also be audited)
    # -----------------------------------------------------------------------
    final = compute_shape_metrics(final)

    # -----------------------------------------------------------------------
    # 10. CANDIDATE FLAG COLUMN
    #     Convenience boolean for quick filtering and visual QA in QGIS/geemap
    # -----------------------------------------------------------------------
    final['needs_review'] = final['label'] == 'candidate'

    # -----------------------------------------------------------------------
    # 11. SUMMARY REPORT
    # -----------------------------------------------------------------------
    print("=" * 55)
    print("  Industrial Buildings Classifier — Summary")
    print("=" * 55)
    label_counts = final['label'].value_counts()
    for lbl, cnt in label_counts.items():
        print(f"  {lbl:<12} : {cnt:>6,}")
    print(f"  {'TOTAL':<12} : {len(final):>6,}")
    print("-" * 55)
    print("  Top confidence notes (candidates):")
    cand_notes = (
        final[final['label'] == 'candidate']['confidence_note']
        .value_counts().head(5)
    )
    for note, cnt in cand_notes.items():
        print(f"    {note:<35} : {cnt:>5,}")
    print("=" * 55)

    return final

### Step 4: Land Use classification to help filter warehouses

In [ ]:
# ==============================================================================
# LAND USE CLASSIFICATION LOOKUP TABLES
# Based on landuse_gr column values (consistent across all county shapefiles)
# ==============================================================================

# Land use classes that EXCLUDE a building from the warehouse inventory
LU_EXCLUDE_CLASSES = {
    'Residential',    # Single/multi-family housing — no warehouses
    'Commercial',     # Retail storefronts, strip malls — not WAIRE-regulated
    'Institutional',  # Schools, hospitals, civic — already caught by name rules
                      # but this adds a spatial confirmation layer
}

# Land use classes that CONFIRM or SUPPORT industrial/warehouse classification
# Note: Transport is included because freight terminals, rail yards, and port
# lands are subject to Rule 2305 if warehouse operations occur on-site
LU_CONFIRM_CLASSES = {
    'Industrial',
    'Transport',
}

# Land use classes treated as NEUTRAL — no LU override applied
# Buildings in these zones keep their existing label unchanged
LU_NEUTRAL_CLASSES = {
    'Agriculture',
    'Bare_land/other',
    'Nature_Vegetation',
    'Water/Wetlands',
}


# ==============================================================================
# STEP 1: LOAD AND MERGE COUNTY LU SHAPEFILES
# ==============================================================================

def load_lu_layers(lu_paths, lu_column='landuse_gr', target_crs=4326):
    """
    Load multiple county-level land use shapefiles and merge into a single
    GeoDataFrame, reprojecting to target CRS if needed.

    Parameters
    ----------
    lu_paths : dict
        Dictionary mapping county label to shapefile path.
        Example:
            {
                'LA': 'data/LA_osm_lulc_training.shp',
                'OR': 'data/OR_osm_lulc_training.shp',
                'RI': 'data/RI_osm_lulc_training.shp',
                'SB': 'data/SB_osm_lulc_training.shp',
            }
    lu_column : str
        Column name containing land use class values (default: 'landuse_gr')
    target_crs : int
        EPSG code to reproject all layers to (default: 4326)

    Returns
    -------
    GeoDataFrame
        Merged LU layer with columns: [lu_column, 'county', 'geometry']
    """
    frames = []

    for county, path in lu_paths.items():
        path = Path(path)
        if not path.exists():
            print(f"  WARNING: File not found for {county}: {path} — skipping")
            continue

        gdf = gdp.read_file(path)

        # Validate column exists
        if lu_column not in gdf.columns:
            raise ValueError(
                f"Column '{lu_column}' not found in {path.name}. "
                f"Available columns: {gdf.columns.tolist()}"
            )

        # Keep only essential columns
        gdf = gdf[[lu_column, 'geometry']].copy()
        gdf['county'] = county

        # Reproject if needed
        if gdf.crs is None:
            print(f"  WARNING: {county} shapefile has no CRS — assuming EPSG:4326")
            gdf = gdf.set_crs(4326)
        elif gdf.crs.to_epsg() != target_crs:
            gdf = gdf.to_crs(target_crs)

        # Validate class values
        unexpected = set(gdf[lu_column].dropna().unique()) - (
            LU_EXCLUDE_CLASSES | LU_CONFIRM_CLASSES | LU_NEUTRAL_CLASSES
        )
        if unexpected:
            print(f"  NOTE: {county} has unexpected landuse_gr values: {unexpected}")
            print(f"        These will be treated as NEUTRAL (no LU override)")

        frames.append(gdf)
        print(f"  Loaded {county}: {len(gdf):,} LU polygons")

    if not frames:
        raise RuntimeError("No LU shapefiles loaded. Check your paths.")

    merged = gdp.GeoDataFrame(
        pd.concat(frames, ignore_index=True),
        crs=target_crs
    )
    print(f"\n  Total LU polygons merged: {len(merged):,}")
    print(f"  Class distribution:\n{merged[lu_column].value_counts().to_string()}\n")

    return merged


# ==============================================================================
# STEP 2: SPATIAL JOIN — ASSIGN LU CLASS TO EACH BUILDING
# ==============================================================================

def assign_lu_to_buildings(buildings_gdf, lu_gdf, lu_column='landuse_gr',
                            use_centroid=True):
    """
    Spatially join land use polygons to building footprints.

    Uses building centroids by default for a clean one-to-one join.
    Falls back to largest-overlap for buildings whose centroid misses all
    LU polygons (can happen at county boundary edges).

    Parameters
    ----------
    buildings_gdf : GeoDataFrame   Building footprints (any label/source)
    lu_gdf        : GeoDataFrame   Merged LU layer from load_lu_layers()
    lu_column     : str            LU class column name
    use_centroid  : bool           Use centroid for join (default True)

    Returns
    -------
    GeoDataFrame
        Input buildings with new column 'lu_class' added.
        NaN in 'lu_class' means no LU polygon was matched (treated as neutral).
    """
    work = buildings_gdf.copy().to_crs(3310)
    lu_proj = lu_gdf.to_crs(3310)

    if use_centroid:
        # --- Primary join: centroid-based (fast, one-to-one) ---
        centroids = work.copy()
        centroids['geometry'] = work.geometry.centroid

        joined = gdp.sjoin(
            centroids[['geometry']],
            lu_proj[[lu_column, 'geometry']],
            how='left',
            predicate='within'
        )

        # Handle duplicates: building centroid inside multiple LU polygons
        # (can happen at polygon boundaries) — industrial wins
        if joined.index.duplicated().any():
            # Sort so industrial/transport come first, then drop duplicates
            priority_order = list(LU_CONFIRM_CLASSES) + \
                             list(LU_NEUTRAL_CLASSES) + \
                             list(LU_EXCLUDE_CLASSES)
            joined[lu_column] = pd.Categorical(
                joined[lu_column],
                categories=priority_order,
                ordered=True
            )
            joined = (joined
                      .sort_values(lu_column)
                      .loc[~joined.index.duplicated(keep='first')])

        work['lu_class'] = joined[lu_column]

        # --- Fallback: largest-overlap for unmatched buildings ---
        unmatched_idx = work[work['lu_class'].isna()].index
        if len(unmatched_idx) > 0:
            print(f"  Running overlap fallback for {len(unmatched_idx):,} "
                  f"unmatched buildings...")
            unmatched = work.loc[unmatched_idx].copy()
            overlap_join = gdp.sjoin(
                unmatched[['geometry']],
                lu_proj[[lu_column, 'geometry']],
                how='left',
                predicate='intersects'
            )
            if not overlap_join.empty:
                # Keep only the LU class with the largest intersection area
                overlap_join = overlap_join.reset_index()
                overlap_join = overlap_join.rename(
                    columns={'index': 'bldg_idx'}
                )
                # Compute intersection areas
                def _max_overlap_class(group):
                    bldg_geom = work.loc[group['bldg_idx'].iloc[0], 'geometry']
                    best_class = None
                    best_area  = 0
                    for _, row in group.iterrows():
                        lu_idx = row.get('index_right')
                        if pd.isna(lu_idx):
                            continue
                        lu_geom = lu_proj.iloc[int(lu_idx)].geometry
                        try:
                            inter = bldg_geom.intersection(lu_geom).area
                            if inter > best_area:
                                best_area  = inter
                                best_class = row[lu_column]
                        except Exception:
                            continue
                    return best_class

                fallback_classes = (overlap_join
                                    .groupby('bldg_idx')
                                    .apply(_max_overlap_class))
                work.loc[fallback_classes.index, 'lu_class'] = \
                    fallback_classes.values

    return work.to_crs(4326)


# ==============================================================================
# STEP 3: APPLY LU FILTER — UPDATE LABELS
# ==============================================================================

def apply_lu_filter(buildings_gdf, lu_gdf, lu_column='landuse_gr'):
    """
    Apply land use filter to the classified buildings GeoDataFrame.

    Rules
    -----
    1. Building in CONFIRM zone (Industrial/Transport)
         → label upgraded to 'confirmed' if it was 'candidate'
         → confidence_note updated to reflect LU confirmation
         → already-confirmed buildings: note appended, label unchanged

    2. Building in EXCLUDE zone (Residential/Commercial/Institutional)
         AND no CONFIRM zone overlap
         → label set to 'excluded'
         → confidence_note updated to 'lu_exclude:<class>'

    3. Building overlapping BOTH confirm AND exclude zones
         → Industrial wins: treated as Rule 1 above

    4. Building in NEUTRAL zone or no LU match (NaN)
         → label unchanged
         → 'lu_class' column added for reference

    Parameters
    ----------
    buildings_gdf : GeoDataFrame   Output from industrial_buildings()
    lu_gdf        : GeoDataFrame   Merged LU layer from load_lu_layers()
    lu_column     : str            LU class column name

    Returns
    -------
    GeoDataFrame
        Updated buildings with 'lu_class' and updated 'label',
        'confidence_note', and 'needs_review' columns.
    """

    print("=" * 55)
    print("  Land Use Filter — Starting")
    print("=" * 55)

    # --- Assign LU class to each building ---
    work = assign_lu_to_buildings(buildings_gdf, lu_gdf, lu_column)

    # Track label changes for reporting
    label_before = work['label'].copy()

    # --- Apply rules row by row ---
    def _apply_lu_rule(row):
        lu     = row.get('lu_class')
        label  = row.get('label', 'candidate')
        note   = row.get('confidence_note', '')

        # No LU match → neutral, no change
        if pd.isna(lu) or lu not in (
            LU_EXCLUDE_CLASSES | LU_CONFIRM_CLASSES
        ):
            return label, note

        # Confirm zone → upgrade candidate, append note to confirmed
        if lu in LU_CONFIRM_CLASSES:
            if label == 'candidate':
                return 'confirmed', f'lu_confirm:{lu}'
            else:
                # Already confirmed — append LU note
                return label, f'{note}|lu_confirm:{lu}'

        # Exclude zone → only exclude if not already confirmed
        # (industrial wins: if building was confirmed by name/tag, keep it)
        if lu in LU_EXCLUDE_CLASSES:
            if label == 'confirmed':
                # Strong positive signal overrides LU exclusion
                # Flag it so you can audit the conflict
                return label, f'{note}|lu_conflict_kept:{lu}'
            else:
                return 'excluded', f'lu_exclude:{lu}'

        return label, note

    results = work.apply(_apply_lu_rule, axis=1, result_type='expand')
    results.columns = ['label', 'confidence_note']
    work['label']           = results['label']
    work['confidence_note'] = results['confidence_note']

    # Refresh needs_review flag
    work['needs_review'] = work['label'] == 'candidate'

    # --- Summary report ---
    label_after = work['label']
    changed = (label_before != label_after).sum()

    print(f"\n  Labels updated: {changed:,} buildings changed")
    print("\n  Final label distribution:")
    for lbl, cnt in label_after.value_counts().items():
        print(f"    {lbl:<12} : {cnt:>6,}")

    print("\n  LU class distribution across buildings:")
    for lu_cls, cnt in work['lu_class'].value_counts(dropna=False).items():
        print(f"    {str(lu_cls):<25} : {cnt:>6,}")

    # Audit: confirmed buildings that conflicted with LU exclusion zones
    conflicts = work[work['confidence_note'].str.contains(
        'lu_conflict_kept', na=False
    )]
    if len(conflicts) > 0:
        print(f"\n  AUDIT: {len(conflicts):,} confirmed buildings sit in "
              f"exclusion zones (lu_conflict_kept).")
        print(f"  Review these in QGIS using: "
              f"confidence_note LIKE '%lu_conflict_kept%'")

    print("=" * 55)

    return work


# ==============================================================================
# USAGE EXAMPLE
# ==============================================================================

"""
# 1. Define paths to your county LU shapefiles
lu_paths = {
    'LA': 'path/to/LA_osm_lulc_training.shp',
    'OR': 'path/to/OR_osm_lulc_training.shp',
    'RI': 'path/to/RI_osm_lulc_training.shp',
    'SB': 'path/to/SB_osm_lulc_training.shp',
}

# 2. Load and merge all county LU layers
lu_merged = load_lu_layers(lu_paths)

# 3. Run your existing classifier first
gdf_industrial = industrial_buildings(osm_reduced, ov_buildings, ms_buildings)

# 4. Apply LU filter as a post-processing step
gdf_final = apply_lu_filter(gdf_industrial, lu_merged)

# 5. When Imperial and San Joaquin are ready, just add them:
# lu_paths['IMP'] = 'path/to/IMP_osm_lulc_training.shp'
# lu_paths['SJ']  = 'path/to/SJ_osm_lulc_training.shp'
# lu_merged = load_lu_layers(lu_paths)   # re-run, rest stays identical

# 6. Quick QA filters for QGIS / geemap
confirmed  = gdf_final[gdf_final['label'] == 'confirmed']
candidates = gdf_final[gdf_final['needs_review']]
conflicts  = gdf_final[gdf_final['confidence_note'].str.contains(
                'lu_conflict_kept', na=False)]

# 7. Export
gdf_final.to_file("warehouse_inventory_lu_filtered.gpkg", driver="GPKG")
"""

# Results

In [ ]:
# Download overture data for CA
# One call per county (memory or timeout limits)
# County bounding boxes
bboxes = {
    "la": (-118.9448, 33.7036, -117.6463, 34.8233),
    "or": (-117.945, 33.385, -117.412, 33.927),
    "ri": (-117.7051, 33.4337, -114.4341, 34.1187),
    "sb": (-118.1507, 33.5028, -114.1308, 35.8146),
    "ve": (-119.75, 33.00, -118.63, 34.90),
}

#for county, bbox in bboxes.items():
#    run_overture_download(county, bbox)
#    time.sleep(5)

# Individual call example for one county (uncomment to run)
# ! overturemaps download --bbox=-119.75,33.00,-118.63,34.90 -f geojson --type=building -o ve_buildings.geojson

In [ ]:
# ---------------- Step 1: location ------------------ #
sca_counties = ['Los Angeles', 'Orange', 'Riverside', 'San Bernardino']
roi = get_roi(sca_counties, '06')

# ---------------- Step 2: building ------------------ #
# NAIP
naip_img = get_naip_image(roi, '2022-01-01', '2022-12-31')
naip_buildings = detect_large_roofs(naip_img, roi, area_threshold_ft2=99_000)

## LA
la_overture_gdf = get_overture_building("la_buildings.geojson", area_threshold_ft2=100_000, county="LA")
la_buildings = get_ms_buildings(bboxes["la"], state_name="California", area_threshold_ft2=100_000)
la_osm_buildings = get_osm_buildings("Los Angeles County, California, USA")
## Orange
or_overture_gdf = get_overture_building("or_buildings.geojson", area_threshold_ft2=100_000, county="OR")
or_buildings = get_ms_buildings(bboxes["or"], state_name="California", area_threshold_ft2=100_000)
or_osm_buildings = get_osm_buildings("Orange County, California, USA")
## Riverside
ri_overture_gdf = get_overture_building("ri_buildings.geojson", area_threshold_ft2=100_000, county="RI")
ri_buildings = get_ms_buildings(bboxes["ri"], state_name="California", area_threshold_ft2=100_000)
ri_osm_buildings = get_osm_buildings("Riverside County, California, USA")
## San Bernardino
sb_overture_gdf = get_overture_building("sb_buildings.geojson", area_threshold_ft2=100_000, county="SB")
sb_buildings = get_ms_buildings(bboxes["sb"], state_name="California", area_threshold_ft2=100_000)
sb_osm_buildings = get_osm_buildings("San Bernardino County, California, USA")
## Ventura
ve_overture_gdf = get_overture_building("ve_buildings.geojson", area_threshold_ft2=100_000, county="VE")
ve_buildings = get_ms_buildings(bboxes["ve"], state_name="California", area_threshold_ft2=100_000)
ve_osm_buildings = get_osm_buildings("Ventura County, California, USA")


# ------ Step 3: spatial join and classification------ #
# merge Microsoft buildings for all four counties
ms_buildings = gdp.GeoDataFrame(pd.concat(
    [la_buildings, or_buildings, ri_buildings, sb_buildings],
    ignore_index=True
), crs=4326)

# merge Overture buildings for all four counties
ov_buildings = gdp.GeoDataFrame(pd.concat(
    [la_overture_gdf, or_overture_gdf, ri_overture_gdf, sb_overture_gdf],
    ignore_index=True
), crs=4326)

# merge Overture buildings for all four counties
osm_buildings = gdp.GeoDataFrame(pd.concat(
    [la_osm_buildings, or_osm_buildings, ri_osm_buildings, sb_osm_buildings],
    ignore_index=True
), crs=4326)
columns_to_keep = ['geometry', 'name', 'building', 'tourism', 'amenity', 'source_year', 'area_ft2']
osm_reduced = osm_buildings[columns_to_keep].copy()

# Clean and classify
all_industrial_buildings = industrial_buildings(osm_reduced, ov_buildings, ms_buildings)


# ---------------- Step 4: Land-Use Classification ---------------- #
# 1. Define paths to your county LU shapefiles
lu_paths = {
    'LA': path + 'Shapefiles/LA_osm_lulc_training.shp',
    'OR': path + 'Shapefiles/OR_osm_lulc_training.shp',
    'RI': path + 'Shapefiles/RI_osm_lulc_training.shp',
    'SB': path + 'Shapefiles/SB_osm_lulc_training.shp',
    'VE': path + 'Shapefiles/VE_osm_lulc_training.shp',
}

# 2. Load and merge all county LU layers
lu_merged = load_lu_layers(lu_paths)

# 3. Run your existing classifier first
gdf_industrial = industrial_buildings(osm_reduced, ov_buildings, ms_buildings)

# 4. Apply LU filter as a post-processing step
gdf_final = apply_lu_filter(gdf_industrial, lu_merged)

# Export
gdf_final.to_file(path + "warehouse_inventory_lu_filtered.gpkg", driver="GPKG")

In [ ]:
# Define map centered on Southern California (LA region)
Map = geemap.Map(center=[34.05, -118.25], zoom=8)

# Add merged Microsoft buildings as a vector layer
Map.add_gdf(
    ms_buildings.sample(5000),
    layer_name="Microsoft Buildings (≥100k ft²)",
    style={"color": "orange", "fillColor": "none", "weight": 1}
)

# Add merged Overture buildings as a vector layer
Map.add_gdf(
    ov_buildings.sample(5000),
    layer_name="Overture Buildings (≥100k ft²)",
    style={"color": "blue", "fillColor": "none", "weight": 1}
)


# Add merged OSM buildings as a vector layer
Map.add_gdf(
    industrial_buildings,
    layer_name="OSM Buildings (≥100k ft²)",
    style={"color": "Black", "fillColor": "none", "weight": 1}
)

# Show map
Map

In [ ]:
# Define map centered on Southern California (LA region)
Map = geemap.Map(center=[34.05, -118.25], zoom=8)

# Add NAIP 2022 
try:
    Map.addLayer(naip_buildings['naip'], {'bands': ['R', 'G', 'B'], 'min': 0, 'max': 255}, 'NAIP 2022')
    Map.addLayer(naip_buildings['roof_mask'], {'palette': ['yellow']}, 'Roof Mask')
    Map.addLayer(naip_buildings['roof_polygons'].style(color='orange', fillColor='00000000', width=1), 
                 {}, 'Large Roof Polygons')
except NameError:
    print("No NAIP image found — skipping base layer.")

# Add region of interest if available
try:
    Map.addLayer(roi, {}, 'ROI')
except NameError:
    print("No ROI defined — skipping ROI layer.")

# Add merged Microsoft buildings as a vector layer
Map.add_gdf(
    ms_buildings.sample(5000),
    layer_name="Microsoft Buildings (≥100k ft²)",
    style={"color": "orange", "fillColor": "none", "weight": 1}
)

# Add merged Overture buildings as a vector layer
Map.add_gdf(
    ov_buildings.sample(5000),
    layer_name="Overture Buildings (≥100k ft²)",
    style={"color": "blue", "fillColor": "none", "weight": 1}
)

# Add merged OSM buildings as a vector layer
Map.add_gdf(
    osm_buildings,
    layer_name="OSM Buildings (≥100k ft²)",
    style={"color": "Black", "fillColor": "none", "weight": 1}
)

# Show map
Map